In [1]:
# import required libs
import numpy as np
import sys
sys.path.append(r"../Communiaction")
sys.path.append(r"../Utils")
import TCP_IP_UTILS
import Utils
import time

In [2]:
#connect to ethernet socket 
HOST = '192.168.1.10'  # Remote device IP (server IP)
PORT = 7               # Echo port
client_socket = TCP_IP_UTILS.tcp_connect(HOST,PORT)
#Set IOs to default before turning on powersupply
ret_data = Utils.Set_ChipIO_to_Default(client_socket)
print(ret_data)
time.sleep(1) #Wait for IOs to settle

[Client] Connected to 192.168.1.10:7
Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 1
Received Data Size: 1
All Data Received
Default Signals Set
[128]


In [ ]:
#Add power supply code here

In [3]:
#Create Checker pattern for test. Make sure calib supplys are off.
rows, cols = 296, 1156
full_array = np.zeros(shape=(320, 1156),dtype=np.uint8) #320 is the padded list.
# Step 3: Insert checkerboard into left part (first 296 columns)
full_array = full_array.flatten(order='F') #Across Column

In [4]:
#Call Write SRAM here
write_data = full_array.tolist()  #Remember this is in bits. Will be converted to bytes inside
ret_data = Utils.Write_SRAM(client_socket,write_data,1)
print(ret_data)

Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 6
Received Data Size: 1
All Data Received
[128]


In [5]:
import numpy as np
import random
import re
def scan_gen(time):
    SC_data = f"""
    IN_EXT_LOC = 1'b0;
    TDC_EN_EXT_LOC = 1'b0;
    ENCHG_EXT_LOC = 1'b0;
    ENTIME_EXT_LOC = 1'b0;
    PCHG_EXT_LOC = 1'b0;
    VDAC_CTRL_EXT_LOC = 1'b0;
    DEL_RST_EXT_LOC = 1'b0;
    TDC_COMPUTE_EXT_LOC = 1'b0;
    TDC_RST_EXT_LOC = 1'b0;
    VTC_EN_EXT_LOC = 1'b0;


    IN_LB = 6'd32;
    IN_UB = 6'd35;
    TDC_EN_LB = 6'd32;
    TDC_EN_UB = 6'd38;
    ENCHG_LB = 6'd32;
    ENCHG_UB = 6'd41;
    ENTIME_LB = 6'd32;
    ENTIME_UB = 6'd44;
    PCHG_LB = 6'd32;
    PCHG_UB = 6'd47;
    VDAC_CTRL_LB = 6'd32;
    VDAC_CTRL_UB = 6'd50;
    DEL_RST_LB = 6'd32;
    DEL_RST_UB = 6'd52;
    TDC_COMPUTE_LB = 6'd32;
    TDC_COMPUTE_UB = 6'd54;
    TDC_RST_LB = 6'd32;
    TDC_RST_UB = 6'd54;
    BL_NUM_TGT = 2'd0;
    BL_DONE_TIME = 6'd62;
    VTC_EN_LB = 6'd32;
    VTC_EN_UB = 6'd54;
    SAMPLE_EDGE_TIME = 6'd{time};

    CHG_TIME = 1'd0;
    OSC_DEL = 1'd0;
    """
    SC_order = ["BL_DONE_TIME","SAMPLE_EDGE_TIME", "BL_NUM_TGT", "BLANK_4", "IN_LB", "TDC_EN_LB", "ENCHG_LB", "IN_EXT_LOC","TDC_EN_EXT_LOC","ENCHG_EXT_LOC",\
                "ENTIME_EXT_LOC","PCHG_EXT_LOC","VDAC_CTRL_EXT_LOC", 'ENTIME_LB', 'PCHG_LB', 'VDAC_CTRL_LB', 'DEL_RST_LB', 'TDC_COMPUTE_LB', 'TDC_RST_LB', \
                'VTC_EN_LB',"DEL_RST_EXT_LOC","TDC_COMPUTE_EXT_LOC","TDC_RST_EXT_LOC","VTC_EN_EXT_LOC","CHG_TIME","OSC_DEL", "VTC_EN_UB","TDC_RST_UB","TDC_COMPUTE_UB",\
                "DEL_RST_UB","VDAC_CTRL_UB","PCHG_UB","ENTIME_UB","ENCHG_UB","TDC_EN_UB","IN_UB","BLANK_12_PISO"]

    lines = SC_data.splitlines()
    sc_signal = {}
    for line in lines:
        if line.strip():  # Skip empty lines
            # Handle 'd' (decimal) format
            match = re.match(r"(\w+) = (\d+)'d([0-9]+);", line.strip())
            if match:
                signal_name = match.group(1)
                num_bits = int(match.group(2))  # Number of bits
                signal_value = match.group(3)  # Decimal value


                # If there's only 1 bit, do not add the bit-range
                if num_bits == 1:
                    name = signal_name
                else:
                    # Format the name with the bit-range (e.g., VTC_EN_UB<[5:0]>)
                    name = "%s" % (signal_name) # Using 0-based index for the range

                # Format the decimal value to binary and pad to the correct number of bits
                binary_value = bin(int(signal_value))[2:].zfill(num_bits)
                sc_signal[name] = binary_value


            # Handle 'b' (binary) format
            elif "b" in line:  # e.g., BL_NUM_TGT = 2'b01;
                match = re.match(r"(\w+) = (\d+)'b([01]+);", line.strip())
                if match:
                    signal_name = match.group(1)
                    num_bits = int(match.group(2))  # Number of bits
                    signal_value = match.group(3)  # Binary value

                    # If there's only 1 bit, do not add the bit-range
                    if num_bits == 1:
                        name = signal_name
                    else:
                        # Format the name with the bit-range (e.g., BL_NUM_TGT<[1:0]> for 2 bits)
                        name = "%s" % (signal_name)

                    # Format the binary value to match the number of bits
                    binary_value = signal_value.zfill(num_bits)
                    sc_signal[name] = binary_value


    # print(sc_signal)
    output_list = []
    for signal in SC_order:
        if "BLANK" in signal:
            num_blanks = re.findall(r'\d+', signal)
            output_list += [0]*int(num_blanks[0])
        else:
            temp = sc_signal[signal][::-1]
            output_list += list(map(int,temp))
    print(sc_signal["SAMPLE_EDGE_TIME"])
    output_list = output_list[::-1] + [0]*30
    return output_list

In [7]:
control_data = scan_gen(62)

111110


In [8]:
#Call Scan IN here with data
scan_id = Utils.CONTROL_LOGIC_SC["id"]
scan_len = Utils.CONTROL_LOGIC_SC["len"]
scan_data = control_data
ret_data = Utils.ScanIN_Data(client_socket,scan_id,scan_len,scan_data,1)
print(ret_data)
#Set IOs to default before turning on powersupply
#ret_data = Utils.Set_ChipIO_to_Default(client_socket)
#print(ret_data)

[162, 0]
Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 3
Received Data Size: 1
All Data Received
[128]


In [9]:
signal_array = [Utils.CTRL_EN]
value_array = [1]
ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,1)
print(ret_data)

Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 1
Received Data Size: 1
All Data Received
[128]


In [10]:
signal_array = [Utils.DFF_RST]
value_array = [1]
ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,1)
print(ret_data)

Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 1
Received Data Size: 1
All Data Received
[128]


In [11]:
signal_array = [Utils.CTRL_EN]
value_array = [0]
ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,1)
print(ret_data)

Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 1
Received Data Size: 1
All Data Received
[128]


In [12]:

signal_array = [Utils.SCN_SEL_0,Utils.SCN_SEL_1,Utils.SCN_SEL_2,Utils.CLK_A,Utils.CLK_B,Utils.SCN_IN]
value_array = [1,1,1,0,0,0]
ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,1)

signal_array = [Utils.IN_EN]
value_array = [1]
ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,1)
print(ret_data)
signal_array = [Utils.CLK_A]
value_array = [1]
ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,1)
print(ret_data)
signal_array = [Utils.CLK_A]
value_array = [0]
ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,1)
print(ret_data)
signal_array = [Utils.IN_EN]
value_array = [0]
ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,1)
print(ret_data)

Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 1
Received Data Size: 1
All Data Received
Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 1
Received Data Size: 1
All Data Received
[128]
Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 1
Received Data Size: 1
All Data Received
[128]
Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 1
Received Data Size: 1
All Data Received
[128]
Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 1
Received Data Size: 1
All Data Received
[128]


In [13]:
#Call Scan OUT here with data
scan_id = Utils.CONTROL_LOGIC_SC["id"]
scan_len = Utils.CONTROL_LOGIC_SC["len"]
scan_out_data = Utils.ScanOUT_Data(client_socket,scan_id,scan_len,1)
print(len(scan_out_data))
print(scan_out_data)

[162, 0]
Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 4
Received Data Size: 24
All Data Received
24
[0, 128, 179, 178, 233, 167, 101, 219, 54, 32, 8, 130, 32, 8, 2, 32, 8, 2, 224, 251, 0, 0, 0, 0]


In [14]:
ret_data = Utils.Set_ChipIO_to_Default(client_socket)
print(ret_data)

Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 1
Received Data Size: 1
All Data Received
Default Signals Set
[128]


In [15]:
u8_array = []
for i in range(0, len(scan_data), 8):
    byte_bits = scan_data[i:i+8]
    value = sum(bit << j for j, bit in enumerate(byte_bits))  # index 0 is LSB
    u8_array.append(value)

print(u8_array)
print(scan_out_data == u8_array)

[0, 16, 103, 101, 211, 79, 203, 182, 109, 64, 16, 4, 65, 16, 4, 64, 16, 4, 192, 247, 1, 0, 0, 0]
False


In [ ]:
signal_array = [Utils.BL_SEL_0,Utils.BL_SEL_1]
value_array = [0,0]
ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,1)
print(ret_data)